# 11.7 — Eligibility Traces & TD(λ)

Eligibility traces let a temporal-difference learner remember recently visited states, so one surprise can update a whole trail of earlier decisions instead of only the most recent state. In this lesson, `λ` controls how far that credit flows backward: `λ=0` behaves like one-step TD, `λ≈1` behaves more like Monte Carlo returns, and the useful middle ground blends bootstrapping with delayed evidence.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build eligibility traces and TD(λ) one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the forward view, backward view, and trace decay are not black boxes. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays for rewards, values, and traces.
import matplotlib.pyplot as plt  # small visualizations for returns and trace decay.
np.random.seed(0)  # reproducibility for any stochastic examples.

### 1. One-step TD error: the local surprise signal

Temporal-difference learning starts with the smallest possible backup. After seeing reward `r` and next state `s'`, it compares the current value `V(s)` with the bootstrap target `r + γV(s')`. The difference

$$\delta_t=r_{t+1}+\gamma V(S_{t+1})-V(S_t)$$

is the **TD error**: a signed surprise telling us whether the state looked too pessimistic or too optimistic.

In [ ]:
V_w = np.array([0.20, 0.50, 0.80, 0.00])  # current value table for states 0,1,2,terminal.
gamma_w = 0.90  # discount future values by 10% per step.
s_w, next_w, reward_w = 1, 2, 1.0  # transition: state 1 -> state 2 with reward 1.
target_w = reward_w + gamma_w * V_w[next_w]  # one-step bootstrap target.
delta_w = target_w - V_w[s_w]  # TD error: target minus current estimate.
print("target:", round(target_w, 3), "delta:", round(delta_w, 3))
assert round(target_w, 3) == 1.720 and round(delta_w, 3) == 1.220

▶ What you'll see: the target is `1.720`, so state 1 receives a positive TD error of `1.220`.

In [ ]:
alpha_w = 0.40  # learning rate: move 40% of the way toward the target.
V_after_w = V_w.copy()  # keep the original table visible.
V_after_w[s_w] += alpha_w * delta_w  # one-step TD updates only the current state.
print("before:", V_w)
print("after :", np.round(V_after_w, 3))
assert round(V_after_w[s_w], 3) == 0.988

▶ What you'll see: only state 1 changes, from `0.500` to `0.988`.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["V(s)", "target", "updated V(s)"], [V_w[s_w], target_w, V_after_w[s_w]], color=["gray", "black", "seagreen"])
plt.title("1: TD moves partway toward a bootstrap target")
plt.ylabel("value")
plt.show()

▶ What you'll see: the updated value lands between the old estimate and the target, not all the way at the target.

*Why it's done this way:* TD uses `r + γV(s')` because the next value estimate is a compressed summary of all later consequences. The update is incremental because the target is noisy and partly self-referential; moving by `αδ` improves the estimate without letting one transition overwrite the entire table.

### 2. Eligibility traces: a decaying memory of recent states

A one-step TD error knows what just happened, but it does not know who deserves credit. Eligibility traces add a memory vector `e`: when a state is visited its trace increases, and every step all traces decay by `γλ`. Recent states stay eligible; older states fade. The accumulating trace rule is

$$e_t(s)=\gamma\lambda e_{t-1}(s)+\mathbf{1}\{S_t=s\}.$$

In [ ]:
states_w = np.array([0, 1, 2, 1])  # visited states before the episode ends.
gamma_w, lam_w = 0.90, 0.80  # trace decay factor is gamma * lambda = 0.72.
trace_w = np.zeros(4)  # one eligibility value per state.
history_w = []  # save the trace vector after each visit.
for s_w in states_w:
    trace_w *= gamma_w * lam_w  # old credit fades before the new state is marked.
    trace_w[s_w] += 1.0  # accumulating trace: every visit adds one unit of eligibility.
    history_w.append(trace_w.copy())
history_w = np.array(history_w)
print(np.round(history_w, 3))
assert round(history_w[2, 0], 3) == 0.518

▶ What you'll see: state 0's trace decays from `1.000` to `0.720` to `0.518` while newer states are larger.

In [ ]:
plt.figure(figsize=(5, 3))
for state_w in range(3):
    plt.plot(history_w[:, state_w], marker="o", label=f"state {state_w}")
plt.title("2: eligibility traces decay by γλ")
plt.xlabel("time step")
plt.ylabel("eligibility")
plt.legend()
plt.show()

▶ What you'll see: each state spikes when visited, then decays geometrically; the revisit to state 1 raises its trace again.

*Why it's done this way:* credit assignment needs recency. Multiplying by `γ` matches the discount used for rewards, and multiplying by `λ` is an extra modeling choice: small `λ` says only the newest state should receive much credit, while large `λ` says earlier states are still plausibly responsible for delayed outcomes.

### 3. Forward view: the λ-return blends n-step targets

The forward view defines what TD(λ) is trying to approximate. From a starting time, it forms every n-step return, then averages them with geometric weights controlled by `λ`. For a finite episode the λ-return can be computed backward as

$$G_t^\lambda = R_{t+1}+\gamma\big((1-\lambda)V(S_{t+1})+\lambda G_{t+1}^\lambda\big),$$

with the terminal value equal to the final reward-only return.

In [ ]:
rewards_w = np.array([0.0, 0.0, 1.0])  # delayed payoff arrives only at the end.
values_w = np.array([0.2, 0.3, 0.4, 0.0])  # values for S0,S1,S2,terminal.
gamma_w, lam_w = 0.90, 0.70  # blend one-step bootstraps with longer evidence.
G_lambda_w = np.zeros(3)  # one lambda-return for each nonterminal time.
next_return_w = 0.0  # terminal continuation is zero after the final reward.
for t_w in range(2, -1, -1):
    bootstrap_w = values_w[t_w + 1]  # V(S_{t+1}) used by the one-step part.
    G_lambda_w[t_w] = rewards_w[t_w] + gamma_w * ((1 - lam_w) * bootstrap_w + lam_w * next_return_w)
    next_return_w = G_lambda_w[t_w]
print("lambda returns:", np.round(G_lambda_w, 3))
assert np.allclose(np.round(G_lambda_w, 3), [0.546, 0.738, 1.000])

▶ What you'll see: the start-state target is `0.546`, between a short bootstrap and the full delayed payoff.

In [ ]:
g1_w = rewards_w[0] + gamma_w * values_w[1]  # one-step target from S0.
g2_w = rewards_w[0] + gamma_w * rewards_w[1] + gamma_w**2 * values_w[2]  # two-step target.
g3_w = rewards_w[0] + gamma_w * rewards_w[1] + gamma_w**2 * rewards_w[2]  # full return.
print("1-step, 2-step, full:", round(g1_w, 3), round(g2_w, 3), round(g3_w, 3))
assert round(g1_w, 3) == 0.270 and round(g3_w, 3) == 0.810

▶ What you'll see: short targets lean on value estimates; the full return waits for the delayed reward.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(["1-step", "2-step", "full", "λ-return"], [g1_w, g2_w, g3_w, G_lambda_w[0]], color=["gray", "gray", "gray", "seagreen"])
plt.title("3: λ-return blends short and long targets")
plt.ylabel("target from S0")
plt.show()

▶ What you'll see: the λ-return sits near the shorter targets when `λ=0.70`, but still includes delayed reward information.

*Why it's done this way:* n-step returns trade bias for variance. One-step TD is biased by the current value table but low variance; Monte Carlo is unbiased for the sampled episode but high variance. The λ-return makes that tradeoff continuous by geometrically down-weighting farther targets instead of choosing one horizon abruptly.

### 4. Backward view: TD(λ) applies one TD error to every eligible state

The backward view is the online algorithm. At each transition, compute one TD error, update the trace vector, and apply the same error to every state in proportion to its eligibility:

$$V(s)\leftarrow V(s)+\alpha\delta_t e_t(s).$$

This is powerful because it does not wait until the episode ends, yet for tabular prediction it closely matches the forward λ-return target.

In [ ]:
episode_states_w = np.array([0, 1, 2])  # S0 -> S1 -> S2 -> terminal.
episode_rewards_w = np.array([0.0, 0.0, 1.0])  # only the terminal transition pays.
V_td_w = np.zeros(4)  # start with no value knowledge.
e_w = np.zeros(4)  # eligibility trace table.
alpha_w, gamma_w, lam_w = 0.50, 0.90, 0.80  # learning, discount, and trace memory.
snapshots_w = []  # store values after each transition.
for t_w, s_w in enumerate(episode_states_w):
    next_s_w = s_w + 1  # terminal is index 3 after state 2.
    delta_w = episode_rewards_w[t_w] + gamma_w * V_td_w[next_s_w] - V_td_w[s_w]
    e_w *= gamma_w * lam_w
    e_w[s_w] += 1.0
    V_td_w += alpha_w * delta_w * e_w
    snapshots_w.append(V_td_w.copy())
print("final V:", np.round(V_td_w[:3], 3))
assert np.allclose(np.round(V_td_w[:3], 3), [0.259, 0.36, 0.5])

▶ What you'll see: the final reward updates state 2 most, but states 1 and 0 also increase through their decayed traces.

In [ ]:
plt.figure(figsize=(5, 3))
plt.imshow(np.array(snapshots_w)[:, :3], cmap="viridis", aspect="auto")
plt.colorbar(label="value")
plt.xticks([0, 1, 2], ["S0", "S1", "S2"])
plt.yticks([0, 1, 2], ["after t0", "after t1", "after t2"])
plt.title("4: one delayed reward flows backward")
plt.show()

▶ What you'll see: values remain zero until the reward arrives, then all recently eligible states brighten.

*Why it's done this way:* the backward view is computationally practical. Instead of storing future rewards to later build a λ-return for every past time, it keeps exactly the sufficient memory needed right now: traces say who was recently responsible, and `δ` says whether the latest consequence was better or worse than expected.

### 5. Accumulating vs replacing traces

When a state is revisited, accumulating traces add another `1`, while replacing traces set that state's eligibility to `1`. Accumulating traces can make repeated states receive very large credit; replacing traces cap recency credit and are often more stable in tasks with loops.

In [ ]:
path_w = np.array([0, 1, 1, 1, 2])  # state 1 repeats several times.
gamma_w, lam_w = 0.90, 0.80
acc_w = np.zeros(3)
rep_w = np.zeros(3)
acc_hist_w, rep_hist_w = [], []
for s_w in path_w:
    acc_w *= gamma_w * lam_w
    rep_w *= gamma_w * lam_w
    acc_w[s_w] += 1.0  # repeated visits add up.
    rep_w[s_w] = 1.0  # repeated visits refresh to one.
    acc_hist_w.append(acc_w.copy())
    rep_hist_w.append(rep_w.copy())
acc_hist_w, rep_hist_w = np.array(acc_hist_w), np.array(rep_hist_w)
print("accumulating state1:", np.round(acc_hist_w[:, 1], 3))
print("replacing state1   :", np.round(rep_hist_w[:, 1], 3))
assert round(acc_hist_w[3, 1], 3) == 2.238 and round(rep_hist_w[3, 1], 3) == 1.000

▶ What you'll see: state 1's accumulating trace grows above `2`, while its replacing trace stays capped at `1`.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(acc_hist_w[:, 1], marker="o", label="accumulating")
plt.plot(rep_hist_w[:, 1], marker="s", label="replacing")
plt.title("5: repeated visits change trace semantics")
plt.xlabel("time step")
plt.ylabel("eligibility for state 1")
plt.legend()
plt.show()

▶ What you'll see: the accumulating curve rises with repeated visits; the replacing curve refreshes but does not grow.

*Why it's done this way:* both traces encode recency, but they answer different credit questions. Accumulating traces count every visit as another reason to assign credit; replacing traces say only the most recent visit matters once a state is active. That modeling choice affects stability whenever trajectories loop through the same state many times.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses a
> handful of small numbers, prints the intermediate values with inline `# ->` results, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · One-step TD error updates one state

TD(λ) still starts with an ordinary one-step TD surprise before traces decide how far that surprise
spreads.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_values = np.array([0.1, 0.4, 1.0, 0.0])
print("values:", t1_values.tolist())  # -> [0.1, 0.4, 1.0, 0.0]
t1_state = 1
print("state:", t1_state)  # -> 1
t1_next_state = 2
print("next state:", t1_next_state)  # -> 2
t1_reward = 0.5
print("reward:", t1_reward)  # -> 0.5
t1_gamma = 0.5
print("gamma:", t1_gamma)  # -> 0.5
t1_target = t1_reward + t1_gamma * t1_values[t1_next_state]
print("TD target:", round(float(t1_target), 3))  # -> 1.0
t1_delta = t1_target - t1_values[t1_state]
print("TD error:", round(float(t1_delta), 3))  # -> 0.6
t1_alpha = 0.25
print("alpha:", t1_alpha)  # -> 0.25
t1_new_value = t1_values[t1_state] + t1_alpha * t1_delta
print("updated state value:", round(float(t1_new_value), 3))  # -> 0.55

plt.figure(figsize=(4.4, 3.0))
plt.bar(["old", "target", "new"], [t1_values[t1_state], t1_target, t1_new_value], color=["gray", "black", "seagreen"])
plt.title("Toy 1 · one-step TD surprise")
plt.ylabel("value")
plt.show()

assert abs(t1_new_value - 0.55) < 1e-12

▶ What you'll see: one value moves from `0.4` to `0.55`, not all the way to the target.

### ✍️ Toy 2 · Eligibility traces decay by γλ

A trace spikes when a state is visited, then fades by the product `gamma * lambda` on later steps.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_path = np.array([0, 1, 0])
print("visited path:", t2_path.tolist())  # -> [0, 1, 0]
t2_gamma = 0.5
print("gamma:", t2_gamma)  # -> 0.5
t2_lambda = 0.8
print("lambda:", t2_lambda)  # -> 0.8
t2_decay = t2_gamma * t2_lambda
print("decay:", round(t2_decay, 3))  # -> 0.4
t2_trace = np.zeros(3)
print("initial trace:", t2_trace.tolist())  # -> [0.0, 0.0, 0.0]
t2_history = []
for t2_state in t2_path:
    t2_trace = t2_trace * t2_decay
    t2_trace[t2_state] = t2_trace[t2_state] + 1.0
    t2_history.append(t2_trace.copy())
t2_history = np.array(t2_history)
print("trace history:", np.round(t2_history, 3).tolist())  # -> [[1.0, 0.0, 0.0], [0.4, 1.0, 0.0], [1.16, 0.4, 0.0]]

plt.figure(figsize=(4.8, 3.0))
plt.plot(t2_history[:, 0], marker="o", label="state 0")
plt.plot(t2_history[:, 1], marker="s", label="state 1")
plt.title("Toy 2 · trace memory fades")
plt.xlabel("visit index")
plt.ylabel("eligibility")
plt.legend()
plt.show()

assert np.allclose(np.round(t2_history[-1], 3), np.array([1.16, 0.4, 0.0]))

▶ What you'll see: state 0's trace is refreshed by the final revisit while state 1 fades to `0.4`.

### ✍️ Toy 3 · λ-return blends short and full targets

The forward view places the λ-return between a short bootstrap target and the full Monte Carlo
return.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_rewards = np.array([0.0, 1.0])
print("rewards:", t3_rewards.tolist())  # -> [0.0, 1.0]
t3_values = np.array([0.2, 0.6, 0.0])
print("values:", t3_values.tolist())  # -> [0.2, 0.6, 0.0]
t3_gamma = 0.5
print("gamma:", t3_gamma)  # -> 0.5
t3_lambda = 0.5
print("lambda:", t3_lambda)  # -> 0.5
t3_g1 = t3_rewards[0] + t3_gamma * t3_values[1]
print("one-step target:", round(float(t3_g1), 3))  # -> 0.3
t3_full = t3_rewards[0] + t3_gamma * t3_rewards[1]
print("full return:", round(float(t3_full), 3))  # -> 0.5
t3_last = t3_rewards[1]
print("last lambda return:", round(float(t3_last), 3))  # -> 1.0
t3_lambda_return = t3_rewards[0] + t3_gamma * ((1.0 - t3_lambda) * t3_values[1] + t3_lambda * t3_last)
print("lambda return:", round(float(t3_lambda_return), 3))  # -> 0.4

plt.figure(figsize=(4.6, 3.0))
plt.bar(["1-step", "λ-return", "full"], [t3_g1, t3_lambda_return, t3_full], color=["gray", "seagreen", "black"])
plt.title("Toy 3 · λ blends horizons")
plt.ylabel("target")
plt.show()

assert abs(t3_lambda_return - 0.4) < 1e-12

▶ What you'll see: the λ-return `0.4` lands exactly between the `0.3` one-step target and the `0.5` full return.

### ✍️ Toy 4 · Backward TD(λ) spreads one reward backward

The backward view keeps traces online, so one final reward updates several recently visited states at
once.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_states = np.array([0, 1, 2])
print("states:", t4_states.tolist())  # -> [0, 1, 2]
t4_rewards = np.array([0.0, 0.0, 1.0])
print("rewards:", t4_rewards.tolist())  # -> [0.0, 0.0, 1.0]
t4_values = np.zeros(4)
print("initial values:", t4_values.tolist())  # -> [0.0, 0.0, 0.0, 0.0]
t4_trace = np.zeros(4)
print("initial trace:", t4_trace.tolist())  # -> [0.0, 0.0, 0.0, 0.0]
t4_gamma = 0.5
print("gamma:", t4_gamma)  # -> 0.5
t4_lambda = 0.5
print("lambda:", t4_lambda)  # -> 0.5
t4_alpha = 0.5
print("alpha:", t4_alpha)  # -> 0.5
t4_trace[0] = t4_trace[0] + 1.0
print("trace after state 0:", np.round(t4_trace, 3).tolist())  # -> [1.0, 0.0, 0.0, 0.0]
t4_trace = t4_trace * t4_gamma * t4_lambda
t4_trace[1] = t4_trace[1] + 1.0
print("trace after state 1:", np.round(t4_trace, 3).tolist())  # -> [0.25, 1.0, 0.0, 0.0]
t4_trace = t4_trace * t4_gamma * t4_lambda
t4_trace[2] = t4_trace[2] + 1.0
print("trace after state 2:", np.round(t4_trace, 3).tolist())  # -> [0.062, 0.25, 1.0, 0.0]
t4_delta = 1.0
print("final TD error:", t4_delta)  # -> 1.0
t4_values = t4_values + t4_alpha * t4_delta * t4_trace
print("final values:", np.round(t4_values, 3).tolist())  # -> [0.031, 0.125, 0.5, 0.0]

plt.figure(figsize=(4.8, 3.0))
plt.bar(["S0", "S1", "S2", "T"], t4_values, color="teal")
plt.title("Toy 4 · one reward updates a trail")
plt.ylabel("value after update")
plt.show()

assert np.allclose(np.round(t4_values, 3), np.array([0.031, 0.125, 0.5, 0.0]))

▶ What you'll see: the newest state gets the largest update, but older states still receive credit.

### ✍️ Toy 5 · Accumulating and replacing traces diverge on loops

Repeated visits can make accumulating traces grow above one, while replacing traces cap the active
state at one.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_path = np.array([1, 1, 1])
print("visited path:", t5_path.tolist())  # -> [1, 1, 1]
t5_gamma = 0.5
print("gamma:", t5_gamma)  # -> 0.5
t5_lambda = 0.8
print("lambda:", t5_lambda)  # -> 0.8
t5_decay = t5_gamma * t5_lambda
print("decay:", round(t5_decay, 3))  # -> 0.4
t5_acc = np.zeros(3)
print("initial accumulating trace:", t5_acc.tolist())  # -> [0.0, 0.0, 0.0]
t5_rep = np.zeros(3)
print("initial replacing trace:", t5_rep.tolist())  # -> [0.0, 0.0, 0.0]
t5_acc_history = []
t5_rep_history = []
for t5_state in t5_path:
    t5_acc = t5_acc * t5_decay
    t5_rep = t5_rep * t5_decay
    t5_acc[t5_state] = t5_acc[t5_state] + 1.0
    t5_rep[t5_state] = 1.0
    t5_acc_history.append(t5_acc.copy())
    t5_rep_history.append(t5_rep.copy())
t5_acc_history = np.array(t5_acc_history)
t5_rep_history = np.array(t5_rep_history)
print("accumulating state 1:", np.round(t5_acc_history[:, 1], 3).tolist())  # -> [1.0, 1.4, 1.56]
print("replacing state 1:", np.round(t5_rep_history[:, 1], 3).tolist())  # -> [1.0, 1.0, 1.0]

plt.figure(figsize=(4.8, 3.0))
plt.plot(t5_acc_history[:, 1], marker="o", label="accumulating")
plt.plot(t5_rep_history[:, 1], marker="s", label="replacing")
plt.title("Toy 5 · repeated visits")
plt.xlabel("visit index")
plt.ylabel("eligibility for state 1")
plt.legend()
plt.show()

assert round(float(t5_acc_history[-1, 1]), 3) == 1.56

▶ What you'll see: accumulating rises to `1.56`, while replacing stays pinned at `1.0`.

## 🛠️ Setup

In [ ]:
import numpy as np  # numerical arrays, vectorized Bellman backups, and small assertions.
import matplotlib.pyplot as plt  # line plots, bars, and heatmaps for trace and value debugging.
np.random.seed(0)  # keep stochastic examples reproducible.

def discounted_return(rewards, gamma):  # compute a finite discounted return from a reward sequence.
    rewards = np.asarray(rewards, dtype=float)  # ensure vector arithmetic is predictable.
    powers = gamma ** np.arange(len(rewards))  # weights 1, gamma, gamma^2, ... for each reward.
    return float(np.sum(powers * rewards))  # sum discounted rewards into one return.

def td_error(V, s, r, next_s, gamma):  # compute the one-step temporal-difference error.
    return float(r + gamma * V[next_s] - V[s])  # target minus current value.

def lambda_returns(rewards, values, gamma, lam):  # compute finite-horizon forward-view lambda returns.
    rewards = np.asarray(rewards, dtype=float)  # one reward per transition.
    values = np.asarray(values, dtype=float)  # value for each state plus terminal value.
    out = np.zeros(len(rewards))  # one lambda-return per starting time.
    g_next = values[-1]  # terminal continuation, usually zero.
    for t in range(len(rewards) - 1, -1, -1):  # work backward so G_{t+1}^lambda is known.
        out[t] = rewards[t] + gamma * ((1 - lam) * values[t + 1] + lam * g_next)  # recursive forward-view target.
        g_next = out[t]  # feed this target into the previous time.
    return out  # return all finite-horizon lambda targets.

def run_td_lambda(states, rewards, n_states, alpha, gamma, lam, replacing=False):  # tabular TD(lambda) from scratch.
    V = np.zeros(n_states + 1)  # include one terminal value slot at the end.
    e = np.zeros(n_states + 1)  # eligibility trace for each state.
    values = []  # snapshots of V after every transition.
    traces = []  # snapshots of e after every transition.
    for t, s in enumerate(states):  # process transitions online.
        next_s = n_states if t == len(states) - 1 else states[t + 1]  # terminal after the final listed state.
        delta = rewards[t] + gamma * V[next_s] - V[s]  # local TD surprise.
        e *= gamma * lam  # decay past eligibility.
        if replacing:  # choose trace semantics.
            e[s] = 1.0  # replacing trace caps the current state at one.
        else:
            e[s] += 1.0  # accumulating trace adds a new visit.
        V += alpha * delta * e  # update all eligible states using the same TD error.
        values.append(V.copy())  # save value trajectory for inspection.
        traces.append(e.copy())  # save trace trajectory for inspection.
    return V, np.array(values), np.array(traces)  # return final values and histories.

## 🟢 Basics (warm-up)

### Basic 1 — Discount a delayed reward

**Goal.** Compute a finite discounted return, because eligibility traces still use the same discounted consequence idea as every RL value method. We build it in 2 steps.

In [ ]:
rewards_b1 = np.array([1.0, 0.0, 2.0])  # rewards arriving now, one step later, and two steps later.
gamma_b1 = 0.90  # future rewards are multiplied by powers of 0.9.
weights_b1 = gamma_b1 ** np.arange(len(rewards_b1))  # discount weights for each reward position.
print("weights:", np.round(weights_b1, 3))  # inspect 1, gamma, gamma^2.

▶ What you'll see: the delayed third reward receives weight `0.81`.

In [ ]:
G_b1 = discounted_return(rewards_b1, gamma_b1)  # sum discounted rewards into one return.
print("discounted return:", round(G_b1, 3))  # inspect 1 + 0.9*0 + 0.9^2*2.
assert round(G_b1, 3) == 2.620  # verify the canonical discounted-return arithmetic.
plt.figure(figsize=(4, 3))
plt.bar(["r0", "γr1", "γ²r2"], rewards_b1 * weights_b1, color="teal")
plt.title("Basic 1: discounted reward contributions")
plt.ylabel("contribution to G")
plt.show()

▶ What you'll see: the delayed reward contributes `1.62`, not the full `2.0`.

👀 Takeaway: discounting lets delayed rewards matter while making farther consequences count less.

### Basic 2 — Build a one-step TD target

**Goal.** Combine an observed reward with a next-state value estimate, because one-step TD bootstraps instead of waiting for the whole episode. We build it in 2 steps.

In [ ]:
V_b2 = np.array([0.4, 0.8, 0.0])  # values for current, next, and terminal states.
r_b2 = 1.0  # observed reward on the transition.
gamma_b2 = 0.90  # discount applied to the next value.
target_b2 = r_b2 + gamma_b2 * V_b2[1]  # one-step target r + gamma V(s').
print("one-step target:", round(target_b2, 3))  # inspect the bootstrap target.
assert round(target_b2, 3) == 1.720  # verify 1 + 0.9*0.8.

▶ What you'll see: the target is `1.720`.

In [ ]:
delta_b2 = target_b2 - V_b2[0]  # TD error compares target to current estimate.
print("TD error:", round(delta_b2, 3))  # inspect the signed surprise.
plt.figure(figsize=(4, 3))
plt.bar(["V(s)", "target", "δ"], [V_b2[0], target_b2, delta_b2], color=["gray", "green", "red"])
plt.title("Basic 2: target minus estimate")
plt.show()

▶ What you'll see: the positive TD error says the current state was undervalued.

👀 Takeaway: TD error is the local surprise signal that later gets spread by traces.

### Basic 3 — Apply a scalar TD update

**Goal.** Move one value estimate toward its TD target, because the learning rate controls how much a single observation can change the table. We build it in 2 steps.

In [ ]:
old_v_b3 = 0.4  # current estimate for one state.
target_b3 = 1.72  # bootstrap target from the previous example.
alpha_b3 = 0.5  # move halfway toward the target.
delta_b3 = target_b3 - old_v_b3  # signed correction.
print("delta:", round(delta_b3, 3))  # inspect the requested movement.

▶ What you'll see: the target is `1.32` above the old value.

In [ ]:
new_v_b3 = old_v_b3 + alpha_b3 * delta_b3  # incremental TD update.
print("new value:", round(new_v_b3, 3))  # inspect halfway movement.
assert round(new_v_b3, 3) == 1.060  # verify 0.4 + 0.5*(1.72 - 0.4).
plt.figure(figsize=(4, 3))
plt.bar(["old", "new", "target"], [old_v_b3, new_v_b3, target_b3], color=["gray", "seagreen", "black"])
plt.title("Basic 3: alpha controls the step")
plt.ylabel("value")
plt.show()

▶ What you'll see: the new estimate lands halfway between old value and target.

👀 Takeaway: `α` damps noisy TD targets by making updates incremental.

### Basic 4 — Decay one eligibility trace

**Goal.** Watch a single trace fade through time, because `γλ` is the memory factor that decides how long credit remains active. We build it in 2 steps.

In [ ]:
gamma_b4 = 0.90  # discount factor.
lam_b4 = 0.80  # trace-decay parameter.
decay_b4 = gamma_b4 * lam_b4  # per-step eligibility multiplier.
trace_b4 = decay_b4 ** np.arange(6)  # trace after zero through five decay steps.
print("decay factor:", round(decay_b4, 3))  # inspect gamma times lambda.
assert round(decay_b4, 3) == 0.720  # verify the geometric multiplier.

▶ What you'll see: each step keeps `72%` of the previous eligibility.

In [ ]:
print("trace path:", np.round(trace_b4, 3))  # inspect geometric decay.
assert round(trace_b4[3], 3) == 0.373  # verify 0.72^3.
plt.figure(figsize=(4, 3))
plt.plot(trace_b4, marker="o", color="purple")
plt.title("Basic 4: trace decay")
plt.xlabel("steps since visit")
plt.ylabel("eligibility")
plt.show()

▶ What you'll see: trace height drops smoothly and geometrically after a visit.

👀 Takeaway: eligibility is a decaying memory, not a permanent record.

### Basic 5 — Accumulate traces along a path

**Goal.** Build the trace vector for a short trajectory, because TD(λ) updates every state with nonzero eligibility. We build it in 2 steps.

In [ ]:
path_b5 = np.array([0, 1, 2])  # three consecutive states.
gamma_b5, lam_b5 = 0.90, 0.80  # trace parameters.
e_b5 = np.zeros(3)  # trace vector for states 0, 1, 2.
hist_b5 = []  # store the trace after each state visit.
for s_b5 in path_b5:
    e_b5 *= gamma_b5 * lam_b5  # decay old traces.
    e_b5[s_b5] += 1.0  # mark the current state as eligible.
    hist_b5.append(e_b5.copy())  # save a snapshot.
hist_b5 = np.array(hist_b5)  # convert snapshots to a matrix.
print(np.round(hist_b5, 3))  # inspect the trace table.

▶ What you'll see: the newest state has trace `1`, while older states are smaller.

In [ ]:
assert np.allclose(np.round(hist_b5[-1], 3), [0.518, 0.720, 1.000])  # verify final trace vector.
plt.figure(figsize=(4, 3))
plt.imshow(hist_b5, cmap="viridis", aspect="auto")
plt.colorbar(label="eligibility")
plt.title("Basic 5: trace vector over time")
plt.xlabel("state")
plt.ylabel("time")
plt.show()

▶ What you'll see: diagonal bright cells show visits; older diagonals fade as time passes.

👀 Takeaway: traces turn a trajectory into graded credit weights for recent states.

### Basic 6 — Spread one TD error with traces

**Goal.** Apply the same TD error to several states using their eligibility weights, because this is the core backward-view update. We build it in 2 steps.

In [ ]:
V_b6 = np.zeros(3)  # three state values before the update.
e_b6 = np.array([0.5184, 0.72, 1.0])  # trace weights from the prior path.
delta_b6 = 1.0  # a positive surprise arrives at the latest transition.
alpha_b6 = 0.5  # learning rate.
updates_b6 = alpha_b6 * delta_b6 * e_b6  # per-state TD(lambda) increments.
print("updates:", np.round(updates_b6, 3))  # inspect credit spread.

▶ What you'll see: the newest state receives the largest update, but older states also move.

In [ ]:
V_new_b6 = V_b6 + updates_b6  # update all eligible states at once.
print("new values:", np.round(V_new_b6, 3))  # inspect the updated value table.
assert np.allclose(np.round(V_new_b6, 3), [0.259, 0.360, 0.500])  # verify the trace-weighted increments.
plt.figure(figsize=(4, 3))
plt.bar(["S0", "S1", "S2"], updates_b6, color="seagreen")
plt.title("Basic 6: one TD error flows backward")
plt.ylabel("value increment")
plt.show()

▶ What you'll see: a delayed positive surprise updates all recently visited states.

👀 Takeaway: traces convert one local TD error into credit assignment over time.

### Basic 7 — Compare λ = 0 and λ = 1 traces

**Goal.** See the extremes of trace memory, because `λ=0` is one-step TD and large `λ` approaches long-horizon credit assignment. We build it in 2 steps.

In [ ]:
gamma_b7 = 0.90  # hold discount fixed.
steps_b7 = np.arange(6)  # steps after a visit.
trace_zero_b7 = (gamma_b7 * 0.0) ** steps_b7  # lambda zero keeps only the current state.
trace_one_b7 = (gamma_b7 * 1.0) ** steps_b7  # lambda one decays only by discount.
trace_zero_b7[0] = 1.0  # define the immediate trace as one.
print("λ=0:", np.round(trace_zero_b7, 3))  # inspect no-memory trace.
print("λ=1:", np.round(trace_one_b7, 3))  # inspect long-memory trace.

▶ What you'll see: `λ=0` vanishes after one step, while `λ=1` decays slowly by `γ`.

In [ ]:
assert trace_zero_b7[1] == 0.0 and round(trace_one_b7[3], 3) == 0.729  # verify the extremes.
plt.figure(figsize=(4, 3))
plt.plot(trace_zero_b7, marker="o", label="λ=0")
plt.plot(trace_one_b7, marker="s", label="λ=1")
plt.title("Basic 7: trace-memory extremes")
plt.xlabel("steps since visit")
plt.ylabel("eligibility")
plt.legend()
plt.show()

▶ What you'll see: the two curves show the memory knob controlled by `λ`.

👀 Takeaway: `λ` interpolates between very local TD and long delayed credit.

### Basic 8 — Compute a forward λ-return

**Goal.** Calculate a finite λ-return recursively, because the forward view defines the target that backward traces approximate online. We build it in 2 steps.

In [ ]:
rewards_b8 = np.array([0.0, 0.0, 1.0])  # terminal reward arrives late.
values_b8 = np.array([0.2, 0.3, 0.4, 0.0])  # current value estimates for each state plus terminal.
gamma_b8, lam_b8 = 0.90, 0.70  # blend parameter for the lambda-return.
Glam_b8 = lambda_returns(rewards_b8, values_b8, gamma_b8, lam_b8)  # compute all lambda targets.
print("lambda returns:", np.round(Glam_b8, 3))  # inspect targets for S0, S1, S2.
assert np.allclose(np.round(Glam_b8, 3), [0.546, 0.738, 1.000])  # verify the worked recursion.

▶ What you'll see: early states receive targets below the full terminal return because bootstrapping still matters.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["S0", "S1", "S2"], Glam_b8, color="darkorange")
plt.title("Basic 8: forward-view λ-returns")
plt.ylabel("target")
plt.show()

▶ What you'll see: later states have larger targets because they are closer to the terminal reward.

👀 Takeaway: the λ-return is a smoothed mixture of short and long backup targets.

### Basic 9 — Run a tiny TD(λ) episode

**Goal.** Update a value table online over one short episode, because backward-view TD(λ) is the implementable algorithm. We build it in 2 steps.

In [ ]:
states_b9 = np.array([0, 1, 2])  # nonterminal states visited in order.
rewards_b9 = np.array([0.0, 0.0, 1.0])  # delayed terminal reward.
V_b9, values_b9, traces_b9 = run_td_lambda(states_b9, rewards_b9, n_states=3, alpha=0.5, gamma=0.9, lam=0.8)  # run tabular TD(lambda).
print("final values:", np.round(V_b9[:3], 3))  # inspect values for nonterminal states.
assert np.allclose(np.round(V_b9[:3], 3), [0.259, 0.360, 0.500])  # verify the trace-based credit spread.

▶ What you'll see: all three states get value from the final reward, with larger credit nearer the end.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(values_b9[:, :3], marker="o")
plt.title("Basic 9: values during one episode")
plt.xlabel("transition")
plt.ylabel("value")
plt.legend(["S0", "S1", "S2"])
plt.show()

▶ What you'll see: values remain flat until the rewarding transition, then jump according to traces.

👀 Takeaway: TD(λ) learns online but still propagates delayed rewards backward.

### Basic 10 — Visualize the bias-variance knob

**Goal.** Compare λ-return targets across λ values, because λ controls how much we trust bootstrapping versus sampled returns. We build it in 2 steps.

In [ ]:
lams_b10 = np.array([0.0, 0.3, 0.7, 1.0])  # from pure one-step TD to full-return behavior.
rewards_b10 = np.array([0.0, 0.0, 1.0])  # delayed reward.
values_b10 = np.array([0.2, 0.3, 0.4, 0.0])  # bootstrap estimates.
targets_b10 = np.array([lambda_returns(rewards_b10, values_b10, 0.9, lam_b10)[0] for lam_b10 in lams_b10])  # S0 target for each lambda.
print("S0 targets:", np.round(targets_b10, 3))  # inspect the interpolation.
assert round(targets_b10[0], 3) == 0.270 and round(targets_b10[-1], 3) == 0.810  # verify extremes.

▶ What you'll see: `λ=0` uses the one-step target, while `λ=1` reaches the full discounted reward.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(lams_b10, targets_b10, marker="o", color="crimson")
plt.title("Basic 10: λ moves the target horizon")
plt.xlabel("λ")
plt.ylabel("S0 λ-return")
plt.show()

▶ What you'll see: the target rises as λ gives more weight to the delayed terminal reward.

👀 Takeaway: λ is a continuous target-horizon knob.

## 🟡 Easy

### Easy 1 — Compare forward λ-return updates with TD(λ)

**Goal.** Put forward and backward views side by side, because the forward view explains the target and the backward view supplies the online algorithm. We build it in 3 steps.

In [ ]:
states_e1 = np.array([0, 1, 2])  # one simple episode.
rewards_e1 = np.array([0.0, 0.0, 1.0])  # terminal reward.
gamma_e1, lam_e1, alpha_e1 = 0.9, 0.8, 0.5  # shared parameters.
initial_values_e1 = np.zeros(4)  # start from zero values.
print("episode states:", states_e1)  # inspect the trajectory.

▶ What you'll see: the episode is a three-state chain ending in reward.

In [ ]:
G_e1 = lambda_returns(rewards_e1, initial_values_e1, gamma_e1, lam_e1)  # forward-view targets from zero values.
forward_update_e1 = alpha_e1 * (G_e1 - initial_values_e1[:3])  # batch-style forward-view increments.
print("forward λ targets:", np.round(G_e1, 3))  # inspect target values.
print("forward increments:", np.round(forward_update_e1, 3))  # inspect batch updates.

In [ ]:
V_e1, values_e1, traces_e1 = run_td_lambda(states_e1, rewards_e1, n_states=3, alpha=alpha_e1, gamma=gamma_e1, lam=lam_e1)  # run online backward view.
print("backward final values:", np.round(V_e1[:3], 3))  # inspect online result.
plt.figure(figsize=(5, 3))
plt.bar(np.arange(3) - 0.18, forward_update_e1, width=0.36, label="forward update")
plt.bar(np.arange(3) + 0.18, V_e1[:3], width=0.36, label="backward TD(λ)")
plt.xticks([0, 1, 2], ["S0", "S1", "S2"])
plt.title("Easy 1: forward target vs backward online update")
plt.legend()
plt.show()

▶ What you'll see: both views assign more credit to later states, but the online backward view uses TD errors as they arrive.

👀 Takeaway: the forward view is the target definition; the backward view is the efficient online implementation.

### Easy 2 — Train repeated episodes on a chain

**Goal.** Repeat TD(λ) over many identical episodes, because value estimates improve iteratively rather than perfectly after one pass. We build it in 3 steps.

In [ ]:
states_e2 = np.array([0, 1, 2, 3])  # four nonterminal chain states.
rewards_e2 = np.array([0.0, 0.0, 0.0, 1.0])  # reward only when leaving state 3.
V_e2 = np.zeros(5)  # include terminal slot.
alpha_e2, gamma_e2, lam_e2 = 0.25, 0.9, 0.8  # learning parameters.
starts_e2 = []  # track V(S0) after each episode.
print("initial values:", V_e2[:4])  # inspect before training.

▶ What you'll see: all nonterminal states start at zero.

In [ ]:
for episode_e2 in range(30):  # repeat the same episode many times.
    e_e2 = np.zeros_like(V_e2)  # traces reset at episode boundaries.
    for t_e2, s_e2 in enumerate(states_e2):  # process one transition.
        next_s_e2 = 4 if t_e2 == len(states_e2) - 1 else states_e2[t_e2 + 1]  # terminal after final state.
        delta_e2 = rewards_e2[t_e2] + gamma_e2 * V_e2[next_s_e2] - V_e2[s_e2]  # TD error.
        e_e2 *= gamma_e2 * lam_e2  # decay old traces.
        e_e2[s_e2] += 1.0  # mark current state.
        V_e2 += alpha_e2 * delta_e2 * e_e2  # update every eligible state.
    starts_e2.append(V_e2[0])  # record start-state value.
print("trained values:", np.round(V_e2[:4], 3))  # inspect learned chain values.
assert V_e2[0] > 0.55  # concrete check that delayed reward reached the start.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(starts_e2, marker="o", color="seagreen")
plt.title("Easy 2: start value improves over episodes")
plt.xlabel("episode")
plt.ylabel("V(S0)")
plt.show()

▶ What you'll see: the start state's value climbs as repeated delayed rewards propagate backward.

👀 Takeaway: traces help delayed rewards reach early states, but repeated experience still matters.

### Easy 3 — Sweep λ on the same chain

**Goal.** Compare learning speed for different λ values, because longer traces can propagate sparse delayed rewards faster. We build it in 3 steps.

In [ ]:
lams_e3 = np.array([0.0, 0.4, 0.8, 1.0])  # trace-memory settings to compare.
curves_e3 = []  # store V(S0) learning curves.
states_e3 = np.array([0, 1, 2, 3])  # same chain for every lambda.
rewards_e3 = np.array([0.0, 0.0, 0.0, 1.0])  # same terminal reward.
print("lambda grid:", lams_e3)  # inspect sweep values.

▶ What you'll see: the sweep ranges from no trace memory to maximum trace memory.

In [ ]:
for lam_e3 in lams_e3:  # train one value function per lambda.
    V_tmp_e3 = np.zeros(5)  # reset values for fair comparison.
    curve_e3 = []  # record start value for this lambda.
    for episode_e3 in range(20):  # repeat episodes.
        e_tmp_e3 = np.zeros(5)  # reset traces each episode.
        for t_e3, s_e3 in enumerate(states_e3):  # run one episode.
            next_s_e3 = 4 if t_e3 == len(states_e3) - 1 else states_e3[t_e3 + 1]  # next or terminal.
            delta_e3 = rewards_e3[t_e3] + 0.9 * V_tmp_e3[next_s_e3] - V_tmp_e3[s_e3]  # TD error.
            e_tmp_e3 *= 0.9 * lam_e3  # decay eligibility.
            e_tmp_e3[s_e3] += 1.0  # activate current state.
            V_tmp_e3 += 0.25 * delta_e3 * e_tmp_e3  # trace-weighted update.
        curve_e3.append(V_tmp_e3[0])  # record progress.
    curves_e3.append(curve_e3)  # store the curve.
print("final V(S0):", np.round([c_e3[-1] for c_e3 in curves_e3], 3))  # inspect learning speed.

In [ ]:
plt.figure(figsize=(5, 3))
for lam_e3, curve_e3 in zip(lams_e3, curves_e3):
    plt.plot(curve_e3, label=f"λ={lam_e3}")
plt.title("Easy 3: λ changes propagation speed")
plt.xlabel("episode")
plt.ylabel("V(S0)")
plt.legend()
plt.show()

▶ What you'll see: larger λ usually lifts the start value sooner on this delayed-reward chain.

👀 Takeaway: long traces can speed credit assignment when rewards are sparse and delayed.

### Easy 4 — Accumulating versus replacing in a loop

**Goal.** Compare trace semantics on a repeated-state path, because loops can make accumulating traces much larger than replacing traces. We build it in 3 steps.

In [ ]:
path_e4 = np.array([0, 1, 1, 1, 2])  # repeated visits to state 1.
gamma_e4, lam_e4 = 0.9, 0.8  # trace decay settings.
acc_e4 = np.zeros(3)  # accumulating traces.
rep_e4 = np.zeros(3)  # replacing traces.
acc_hist_e4, rep_hist_e4 = [], []  # trace histories.
print("path:", path_e4)  # inspect repeated-state pattern.

▶ What you'll see: state 1 appears three times in a row.

In [ ]:
for s_e4 in path_e4:  # walk through the path.
    acc_e4 *= gamma_e4 * lam_e4  # decay accumulating traces.
    rep_e4 *= gamma_e4 * lam_e4  # decay replacing traces.
    acc_e4[s_e4] += 1.0  # add another visit.
    rep_e4[s_e4] = 1.0  # refresh the current state's trace.
    acc_hist_e4.append(acc_e4.copy())  # save accumulating state.
    rep_hist_e4.append(rep_e4.copy())  # save replacing state.
acc_hist_e4, rep_hist_e4 = np.array(acc_hist_e4), np.array(rep_hist_e4)  # matrices for plotting.
print("state 1 accumulating:", np.round(acc_hist_e4[:, 1], 3))  # inspect growth.
print("state 1 replacing:", np.round(rep_hist_e4[:, 1], 3))  # inspect cap.
assert round(acc_hist_e4[3, 1], 3) == 2.238 and round(rep_hist_e4[3, 1], 3) == 1.000  # verify contrast.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(acc_hist_e4[:, 1], marker="o", label="accumulating")
plt.plot(rep_hist_e4[:, 1], marker="s", label="replacing")
plt.title("Easy 4: repeated-state traces")
plt.xlabel("time")
plt.ylabel("eligibility for state 1")
plt.legend()
plt.show()

▶ What you'll see: accumulating traces rise above one; replacing traces stay capped at one.

👀 Takeaway: replacing traces reduce runaway credit in loops by capping repeated-state eligibility.

### Easy 5 — Inspect trace reset at episode boundaries

**Goal.** Reset traces between episodes, because credit from one trajectory should not leak into an unrelated next episode. We build it in 3 steps.

In [ ]:
states_e5 = np.array([0, 1])  # short episode before terminal.
rewards_e5 = np.array([0.0, 1.0])  # reward at the end.
gamma_e5, lam_e5 = 0.9, 0.8  # trace settings.
trace_end_e5 = []  # store final traces from independent episodes.
print("episode length:", len(states_e5))  # inspect trajectory size.

▶ What you'll see: each episode has two nonterminal visits.

In [ ]:
for episode_e5 in range(3):  # run independent episodes.
    e_e5 = np.zeros(3)  # reset traces at the start of each episode.
    for s_e5 in states_e5:  # visit states in the episode.
        e_e5 *= gamma_e5 * lam_e5  # decay old credit within the episode.
        e_e5[s_e5] += 1.0  # mark current state.
    trace_end_e5.append(e_e5.copy())  # save final trace for this episode.
trace_end_e5 = np.array(trace_end_e5)  # convert to a table.
print("end traces each episode:\n", np.round(trace_end_e5, 3))  # inspect identical resets.
assert np.allclose(trace_end_e5[0], trace_end_e5[1])  # verify no cross-episode leakage.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(trace_end_e5, cmap="viridis", aspect="auto")
plt.colorbar(label="eligibility")
plt.title("Easy 5: traces reset each episode")
plt.xlabel("state")
plt.ylabel("episode")
plt.show()

▶ What you'll see: every episode ends with the same trace pattern because each starts from zero.

👀 Takeaway: traces are within-episode memory and should be cleared when an episode terminates.

## 🔴 Advanced

### Advanced 1 — Forward-view λ-return surface

**Goal.** Visualize how the start-state λ-return changes with both `γ` and `λ`, because discount and trace memory jointly decide the effective target horizon. We build it in 3 steps.

In [ ]:
rewards_a1 = np.array([0.0, 0.0, 1.0])  # delayed reward sequence.
values_a1 = np.array([0.2, 0.3, 0.4, 0.0])  # bootstrap values.
gammas_a1 = np.array([0.5, 0.7, 0.9, 0.99])  # discount settings.
lams_a1 = np.array([0.0, 0.4, 0.8, 1.0])  # trace settings.
surface_a1 = np.zeros((len(gammas_a1), len(lams_a1)))  # rows gamma, columns lambda.
print("grid shape:", surface_a1.shape)  # inspect sweep size.

▶ What you'll see: a small 4×4 grid of target settings.

In [ ]:
for i_a1, gamma_a1 in enumerate(gammas_a1):  # loop over discounts.
    for j_a1, lam_a1 in enumerate(lams_a1):  # loop over lambda values.
        surface_a1[i_a1, j_a1] = lambda_returns(rewards_a1, values_a1, gamma_a1, lam_a1)[0]  # S0 target.
print(np.round(surface_a1, 3))  # inspect the target surface.
assert round(surface_a1[2, 0], 3) == 0.270 and round(surface_a1[2, 3], 3) == 0.810  # verify gamma=.9 extremes.

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.imshow(surface_a1, cmap="viridis", aspect="auto", origin="lower")
plt.colorbar(label="S0 λ-return")
plt.xticks(range(len(lams_a1)), lams_a1)
plt.yticks(range(len(gammas_a1)), gammas_a1)
plt.xlabel("λ")
plt.ylabel("γ")
plt.title("Advanced 1: target horizon surface")
plt.show()

▶ What you'll see: higher γ and higher λ give more weight to the delayed reward, raising the start target.

👀 Takeaway: γ discounts consequences; λ controls how far TD credit uses those consequences before bootstrapping.

### Advanced 2 — Value learning under noisy rewards

**Goal.** Compare λ values when terminal rewards are noisy, because longer returns can propagate faster but may carry more variance. We build it in 3 steps.

In [ ]:
rng_a2 = np.random.default_rng(2)  # local reproducible randomness.
lams_a2 = np.array([0.0, 0.6, 0.95])  # compare low, medium, and long traces.
finals_a2 = []  # final start values from several runs.
print("lambda choices:", lams_a2)  # inspect settings.

▶ What you'll see: three trace-memory settings will face the same noisy task.

In [ ]:
for lam_a2 in lams_a2:  # train one learner for each lambda.
    run_values_a2 = []  # save end values across random seeds.
    for run_a2 in range(30):  # multiple noisy repetitions.
        V_a2 = np.zeros(4)  # three states plus terminal.
        local_a2 = np.random.default_rng(run_a2)  # independent but reproducible reward noise.
        for episode_a2 in range(40):  # train for several episodes.
            e_a2 = np.zeros(4)  # reset traces per episode.
            terminal_reward_a2 = 1.0 + local_a2.normal(0.0, 0.25)  # noisy final payoff.
            rewards_a2 = np.array([0.0, 0.0, terminal_reward_a2])  # same chain with noisy end.
            for t_a2, s_a2 in enumerate([0, 1, 2]):  # process the episode.
                next_s_a2 = 3 if t_a2 == 2 else s_a2 + 1  # next state or terminal.
                delta_a2 = rewards_a2[t_a2] + 0.9 * V_a2[next_s_a2] - V_a2[s_a2]  # TD error.
                e_a2 *= 0.9 * lam_a2  # trace decay.
                e_a2[s_a2] += 1.0  # current eligibility.
                V_a2 += 0.2 * delta_a2 * e_a2  # update all eligible states.
        run_values_a2.append(V_a2[0])  # record learned start value.
    finals_a2.append(run_values_a2)  # store distribution for this lambda.
means_a2 = np.array([np.mean(x_a2) for x_a2 in finals_a2])  # average learned values.
stds_a2 = np.array([np.std(x_a2) for x_a2 in finals_a2])  # variability across noisy runs.
print("means:", np.round(means_a2, 3), "stds:", np.round(stds_a2, 3))  # inspect bias/variance behavior.
assert np.all(means_a2 > 0.0)  # verify learning happened for every lambda.

In [ ]:
plt.figure(figsize=(5, 3))
plt.errorbar(lams_a2, means_a2, yerr=stds_a2, marker="o", capsize=4, color="darkorange")
plt.title("Advanced 2: λ with noisy terminal rewards")
plt.xlabel("λ")
plt.ylabel("learned V(S0)")
plt.show()

▶ What you'll see: larger λ can learn stronger delayed credit, with variability shown by error bars.

👀 Takeaway: λ is not always “bigger is better”; it balances speed, bootstrapping bias, and sampled-return variance.

### Advanced 3 — Diagnose an unstable learning rate

**Goal.** Show that traces magnify updates when the learning rate is too large, because one TD error can move many states at once. We build it in 3 steps.

In [ ]:
alphas_a3 = np.array([0.2, 1.2])  # stable and intentionally aggressive step sizes.
curves_a3 = []  # value-magnitude curves.
states_a3 = np.array([0, 1, 2])  # short chain.
rewards_a3 = np.array([0.0, 0.0, 1.0])  # delayed reward.
print("alphas:", alphas_a3)  # inspect step sizes.

▶ What you'll see: the second learning rate is deliberately large for a trace-based update.

In [ ]:
for alpha_a3 in alphas_a3:  # run each learning rate.
    V_a3 = np.zeros(4)  # reset values.
    mags_a3 = []  # maximum absolute value after each episode.
    for episode_a3 in range(12):  # repeat the same episode.
        e_a3 = np.zeros(4)  # reset trace.
        for t_a3, s_a3 in enumerate(states_a3):  # process the chain.
            next_s_a3 = 3 if t_a3 == 2 else states_a3[t_a3 + 1]  # next state or terminal.
            delta_a3 = rewards_a3[t_a3] + 0.9 * V_a3[next_s_a3] - V_a3[s_a3]  # TD error.
            e_a3 *= 0.9 * 0.95  # long trace.
            e_a3[s_a3] += 1.0  # current state.
            V_a3 += alpha_a3 * delta_a3 * e_a3  # update all eligible states.
            V_a3 = np.clip(V_a3, -10, 10)  # keep the diagnostic finite for plotting.
        mags_a3.append(np.max(np.abs(V_a3[:3])))  # track magnitude.
    curves_a3.append(mags_a3)  # save curve.
print("final magnitudes:", [round(c_a3[-1], 3) for c_a3 in curves_a3])  # inspect stability difference.
assert curves_a3[1][-1] >= curves_a3[0][-1]  # concrete check that aggressive steps amplify values.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(curves_a3[0], marker="o", label="α=0.2")
plt.plot(curves_a3[1], marker="s", label="α=1.2")
plt.title("Advanced 3: large α can overshoot with traces")
plt.xlabel("episode")
plt.ylabel("max |V|")
plt.legend()
plt.show()

▶ What you'll see: the aggressive learning rate grows or oscillates much more than the stable one.

👀 Takeaway: because traces update many states per transition, TD(λ) often needs careful step-size tuning.

### Advanced 4 — Off-policy traces with importance ratios

**Goal.** Sketch off-policy correction from scratch, because traces become risky when the data policy and target policy disagree. We build it in 3 steps.

In [ ]:
behavior_probs_a4 = np.array([0.5, 0.5, 0.5])  # behavior policy probability of the sampled action at each step.
target_probs_a4 = np.array([0.9, 0.2, 0.8])  # target policy probability of those same sampled actions.
rhos_a4 = target_probs_a4 / behavior_probs_a4  # importance sampling ratios π(a|s)/b(a|s).
print("importance ratios:", np.round(rhos_a4, 3))  # inspect correction multipliers.
assert np.allclose(np.round(rhos_a4, 3), [1.8, 0.4, 1.6])  # verify ratios.

▶ What you'll see: some sampled actions are more likely under the target policy, others less likely.

In [ ]:
gamma_a4, lam_a4 = 0.9, 0.8  # trace parameters.
e_plain_a4 = np.zeros(3)  # ordinary trace.
e_off_a4 = np.zeros(3)  # ratio-corrected trace.
plain_hist_a4, off_hist_a4 = [], []  # histories.
for t_a4, rho_a4 in enumerate(rhos_a4):  # process sampled actions.
    e_plain_a4 *= gamma_a4 * lam_a4  # ordinary decay.
    e_plain_a4[t_a4] += 1.0  # ordinary activation.
    e_off_a4 *= gamma_a4 * lam_a4 * rho_a4  # off-policy trace is scaled by action likelihood ratio.
    e_off_a4[t_a4] += rho_a4  # current state-action credit also receives the ratio.
    plain_hist_a4.append(e_plain_a4.copy())  # save ordinary trace.
    off_hist_a4.append(e_off_a4.copy())  # save corrected trace.
plain_hist_a4, off_hist_a4 = np.array(plain_hist_a4), np.array(off_hist_a4)  # matrices.
print("final plain:", np.round(plain_hist_a4[-1], 3))  # inspect ordinary credit.
print("final off-policy:", np.round(off_hist_a4[-1], 3))  # inspect ratio-scaled credit.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(3) - 0.18, plain_hist_a4[-1], width=0.36, label="plain")
plt.bar(np.arange(3) + 0.18, off_hist_a4[-1], width=0.36, label="ratio-scaled")
plt.xticks([0, 1, 2], ["step0", "step1", "step2"])
plt.title("Advanced 4: off-policy ratios reshape traces")
plt.ylabel("eligibility")
plt.legend()
plt.show()

▶ What you'll see: importance ratios can shrink or amplify trace credit dramatically.

👀 Takeaway: off-policy TD(λ) needs correction or truncation because traces can become high-variance when policies differ.

### Advanced 5 — Choose λ by validation error

**Goal.** Select λ on held-out transitions, because the best trace horizon is an empirical bias-variance choice rather than a fixed rule. We build it in 3 steps.

In [ ]:
true_values_a5 = np.array([0.9**3, 0.9**2, 0.9, 1.0])  # ideal values for a deterministic four-state chain with terminal reward 1.
lams_a5 = np.array([0.0, 0.3, 0.6, 0.9, 1.0])  # candidate trace parameters.
errors_a5 = []  # validation RMSE for each lambda.
print("true chain values:", np.round(true_values_a5, 3))  # inspect validation target.

▶ What you'll see: earlier states have smaller true values because the terminal reward is farther away.

In [ ]:
for lam_a5 in lams_a5:  # train one value table per lambda.
    V_a5 = np.zeros(5)  # four nonterminal states plus terminal slot.
    for episode_a5 in range(15):  # limited data makes lambda matter.
        e_a5 = np.zeros(5)  # reset traces.
        for t_a5, s_a5 in enumerate([0, 1, 2, 3]):  # deterministic chain.
            reward_a5 = 1.0 if t_a5 == 3 else 0.0  # reward on final transition.
            next_s_a5 = 4 if t_a5 == 3 else s_a5 + 1  # terminal after final state.
            delta_a5 = reward_a5 + 0.9 * V_a5[next_s_a5] - V_a5[s_a5]  # TD error.
            e_a5 *= 0.9 * lam_a5  # trace decay.
            e_a5[s_a5] += 1.0  # current eligibility.
            V_a5 += 0.25 * delta_a5 * e_a5  # update all eligible states.
    rmse_a5 = float(np.sqrt(np.mean((V_a5[:4] - true_values_a5) ** 2)))  # compare with known target values.
    errors_a5.append(rmse_a5)  # store validation error.
errors_a5 = np.array(errors_a5)  # convert to array.
best_lam_a5 = float(lams_a5[np.argmin(errors_a5)])  # choose lowest validation RMSE.
print("RMSE by λ:", np.round(errors_a5, 3), "best λ:", best_lam_a5)  # inspect model selection.
assert best_lam_a5 in lams_a5  # concrete check that selection is valid.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(lams_a5, errors_a5, marker="o", color="navy")
plt.axvline(best_lam_a5, color="crimson", linestyle="--", label=f"best λ={best_lam_a5}")
plt.title("Advanced 5: choose λ by validation")
plt.xlabel("λ")
plt.ylabel("RMSE to true values")
plt.legend()
plt.show()

▶ What you'll see: one λ gives the lowest value error after a limited training budget.

👀 Takeaway: λ should be tuned for the data regime and task, just like learning rate or regularization.